# Notebook 04 — TargetDiff Generation: EGFR Wild-Type (1M17)

Day 4 goals:
- Generate 1000–1500 molecules conditioned on 1M17 pocket
- Save in batches to Google Drive (crash recovery)
- Quick validity check after each batch


In [ ]:
import os, sys, json
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'external', 'targetdiff'))

POCKET_FILE   = f'{PROJECT_ROOT}/data/pockets/1M17_pocket.pdb'
OUTPUT_SDF    = f'{PROJECT_ROOT}/results/generated/1M17_raw.sdf'
CHECKPOINT    = f'{PROJECT_ROOT}/external/targetdiff/checkpoints/pretrained_diffusion.pt'
SAMPLE_SCRIPT = f'{PROJECT_ROOT}/external/targetdiff/scripts/sample_for_pocket.py'

os.makedirs(f'{PROJECT_ROOT}/results/generated', exist_ok=True)

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Run TargetDiff sampling ──────────────────────────────────────────────────
# Option A: call the official script directly
import subprocess

NUM_SAMPLES = 100   # per batch; repeat this cell ~10-15 times or wrap in a loop
BATCH_ID    = 0     # increment for each run to avoid overwriting

batch_out = f'{PROJECT_ROOT}/results/generated/1M17_batch{BATCH_ID:02d}.sdf'

cmd = [
    'python', SAMPLE_SCRIPT,
    '--pdb_file', POCKET_FILE,
    '--num_samples', str(NUM_SAMPLES),
    '--out_sdf', batch_out,
    '--checkpoint', CHECKPOINT,
]
print('Running:', ' '.join(cmd))
# result = subprocess.run(cmd, capture_output=True, text=True)
# print(result.stdout[-2000:])
# if result.returncode != 0:
#     print('STDERR:', result.stderr[-1000:])
print('(Uncomment subprocess.run to actually execute)')

In [ ]:
# ── Merge all batches into one SDF ──────────────────────────────────────────
import glob
from rdkit import Chem

batch_files = sorted(glob.glob(f'{PROJECT_ROOT}/results/generated/1M17_batch*.sdf'))
print(f'Found {len(batch_files)} batch files')

writer = Chem.SDWriter(OUTPUT_SDF)
total = 0
for bf in batch_files:
    for mol in Chem.SDMolSupplier(bf, removeHs=False):
        if mol is not None:
            writer.write(mol)
            total += 1
writer.close()
print(f'Merged {total} molecules → {OUTPUT_SDF}')

In [ ]:
# ── Quick validity check ────────────────────────────────────────────────────
from src.utils import sdf_to_smiles_list

smiles_list = sdf_to_smiles_list(OUTPUT_SDF)
valid = [s for s in smiles_list if s is not None]
print(f'Total: {len(smiles_list)}  |  Valid: {len(valid)}  |  Validity: {len(valid)/len(smiles_list):.1%}')